# Notebook 1 — Empresas Selecionadas para Análise

**Objetivo:** Extrair da bronze as empresas que atendem aos critérios de elegibilidade para análise financeira e persistir na camada Silver.  
**Input:** `layer_01_bronze.cad_cia_aberta`  
**Output:** `layer_02_silver.n0_empresas_selecionadas`

## 1. Importando Bibliotecas

In [ ]:
import pandas as pd
import datetime
import logging
import os
from sqlalchemy import create_engine, text, inspect
from urllib.parse import quote_plus
from dotenv import load_dotenv

In [ ]:
# Carrega variáveis de ambiente
load_dotenv()

In [ ]:
# Definindo a função para criar a engine do banco de dados
def create_db_engine():
        user = quote_plus(os.getenv('DB_USER', 'postgres'))
        password = quote_plus(os.getenv('DB_PASS', 'password'))
        host = os.getenv('DB_HOST', 'localhost')
        port = os.getenv('DB_PORT', '5432')
        dbname = os.getenv('DB_NAME', 'data_lake')
        
        connection_str = f"postgresql+psycopg2://{user}:{password}@{host}:{port}/{dbname}"
        return create_engine(connection_str)

In [ ]:
# Efetivamente criando a engine
engine = create_db_engine()

## 2. Query de Seleção — critérios de elegibilidade das empresas

> Os filtros abaixo definem o universo de ~214 empresas que o pipeline vai processar de ponta a ponta.  
> Qualquer alteração aqui se propaga automaticamente para todos os notebooks subsequentes.

In [ ]:
query_empresas_selecionadas = '''
    SELECT *
    FROM layer_01_bronze.cad_cia_aberta
    WHERE "SIT" = 'ATIVO'                         -- apenas empresas em situação ativa na CVM
    AND "TP_MERC" = 'BOLSA'                       -- obrigatório: sem cotação, P/L e EV/EBITDA são impossíveis
    AND "SIT_EMISSOR" = 'FASE OPERACIONAL'        -- exclui pré-operacionais e empresas em liquidação
    AND "SETOR_ATIV" NOT LIKE '%Emp. Adm. Part%' -- holdings: estrutura contábil incompatível com o Golden Schema
    AND "SETOR_ATIV" NOT LIKE '%Banc%'            -- bancos: regime prudencial com contabilidade diferente
    AND "SETOR_ATIV" NOT LIKE '%Segurad%'         -- seguradoras: mesma incompatibilidade estrutural
    AND "SETOR_ATIV" NOT LIKE '%Financeira%'      -- financeiras: idem
    AND "SETOR_ATIV" NOT LIKE '%Securitiz%'       -- securitizadoras: idem
    AND "SETOR_ATIV" NOT LIKE '%Adm.%Imóv%'      -- administradoras de imóveis: idem
'''

## 3. Extração e Escrita na Silver

In [ ]:
# 1. Extração e Consolidação em Memória
with engine.connect() as conn:
    df_empresas = pd.read_sql(
        text(query_empresas_selecionadas), 
        con=conn
    )

In [ ]:
df_empresas.head()

In [ ]:
df_empresas.head(1).to_dict()

In [ ]:
df_empresas.to_sql(
    name='n0_empresas_selecionadas',
    schema='layer_02_silver',
    con=engine,
    if_exists='replace',  # recria a tabela a cada execução — garante idempotência
    index=False,
    chunksize=10000,      # escrita em lotes para não sobrecarregar a conexão
    method='multi'        # insert múltiplo: mais eficiente que um INSERT por linha
)